In [1]:
import pandas as pd

In [4]:
# Load only first 100k rows (fast and safe)
df = pd.read_csv("data/data.csv", nrows=100_000, low_memory=False)

# Quick inspection
df.head()

,call_number,unit_id,incident_number,call_type,call_date,watch_date,received_dttm,entry_dttm,dispatch_dttm,response_dttm,...,number_of_alarms,unit_type,unit_sequence_in_call_dispatch,fire_prevention_district,supervisor_district,neighborhoods_analysis_boundaries,rowid,case_location,data_as_of,data_loaded_at
0,230010392,T03,23000066,Gas Leak (Natural and LP Gases),2023-01-01T00:00:00.000,2022-12-31T00:00:00.000,2023-01-01T01:56:38.000,2023-01-01T01:58:36.000,2023-01-01T01:59:03.000,2023-01-01T02:02:27.000,...,1,TRUCK,1,1.0,6.0,Tenderloin,230010392-T03,POINT (-122.41316 37.786728),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000
1,230010428,52,23000073,Medical Incident,2023-01-01T00:00:00.000,2022-12-31T00:00:00.000,2023-01-01T02:10:43.000,2023-01-01T02:12:06.000,2023-01-01T02:13:10.000,2023-01-01T02:13:50.000,...,1,MEDIC,2,10.0,10.0,Bayview Hunters Point,230010428-52,POINT (-122.39453 37.724693),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000
2,230011485,B05,23000257,Alarms,2023-01-01T00:00:00.000,2023-01-01T00:00:00.000,2023-01-01T11:33:00.000,2023-01-01T11:34:23.000,2023-01-01T11:34:29.000,2023-01-01T11:35:00.000,...,1,CHIEF,3,5.0,1.0,Lone Mountain/USF,230011485-B05,POINT (-122.446846 37.777668),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000
3,230011085,86,23000201,Medical Incident,2023-01-01T00:00:00.000,2023-01-01T00:00:00.000,2023-01-01T08:28:08.000,2023-01-01T08:30:11.000,2023-01-01T08:32:00.000,2023-01-01T08:32:03.000,...,1,MEDIC,4,8.0,4.0,Sunset/Parkside,230011085-86,POINT (-122.5007 37.762524),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000
4,230012990,T03,23000486,Medical Incident,2023-01-01T00:00:00.000,2023-01-01T00:00:00.000,2023-01-01T20:30:52.000,2023-01-01T20:31:40.000,2023-01-01T20:32:44.000,NaN,...,1,TRUCK,3,1.0,3.0,Tenderloin,230012990-T03,POINT (-122.41335 37.787663),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000


# Basic checks

In [11]:
# Shape
df.shape

(100000, 36)

In [12]:
# Column names
df.columns

Index(['call_number', 'unit_id', 'incident_number', 'call_type', 'call_date',
       'watch_date', 'received_dttm', 'entry_dttm', 'dispatch_dttm',
       'response_dttm', 'on_scene_dttm', 'transport_dttm', 'hospital_dttm',
       'call_final_disposition', 'available_dttm', 'address', 'city',
       'zipcode_of_incident', 'battalion', 'station_area', 'box',
       'original_priority', 'priority', 'final_priority', 'als_unit',
       'call_type_group', 'number_of_alarms', 'unit_type',
       'unit_sequence_in_call_dispatch', 'fire_prevention_district',
       'supervisor_district', 'neighborhoods_analysis_boundaries', 'rowid',
       'case_location', 'data_as_of', 'data_loaded_at'],
      dtype='str')

In [13]:
# Data types
df.dtypes

call_number                            int64
unit_id                                  str
incident_number                        int64
call_type                                str
call_date                                str
watch_date                               str
received_dttm                            str
entry_dttm                               str
dispatch_dttm                            str
response_dttm                            str
on_scene_dttm                            str
transport_dttm                           str
hospital_dttm                            str
call_final_disposition                   str
available_dttm                           str
address                                  str
city                                     str
zipcode_of_incident                  float64
battalion                                str
station_area                         float64
box                                      str
original_priority                        str
priority  

In [14]:
# Missing values
df.isna().sum()

call_number                              0
unit_id                                  0
incident_number                          0
call_type                                0
call_date                                0
watch_date                               0
received_dttm                            0
entry_dttm                               0
dispatch_dttm                            0
response_dttm                         2425
on_scene_dttm                        20783
transport_dttm                       76180
hospital_dttm                        76410
call_final_disposition                   0
available_dttm                          14
address                                 21
city                                   130
zipcode_of_incident                     41
battalion                                0
station_area                             3
box                                      1
original_priority                        0
priority                                 0
final_prior

In [18]:
# Confirm multiple rows per call
df["call_number"].nunique(), len(df)

(48329, 100000)

In [19]:
df.groupby("call_number").size().describe()

count    48329.000000
mean         2.069151
std          1.441005
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         59.000000
dtype: float64

In [20]:
# Compare different response time definitions

df["received_dttm"] = pd.to_datetime(df["received_dttm"])
df["dispatch_dttm"] = pd.to_datetime(df["dispatch_dttm"])
df["on_scene_dttm"] = pd.to_datetime(df["on_scene_dttm"])

df["rt_received"] = (df["on_scene_dttm"] - df["received_dttm"]).dt.total_seconds()
df["rt_dispatch"] = (df["on_scene_dttm"] - df["dispatch_dttm"]).dt.total_seconds()

In [21]:
df[["rt_received", "rt_dispatch"]].describe()

,rt_received,rt_dispatch
count,79217.000000,79217.000000
mean,709.197596,470.806342
std,998.236882,439.312722
min,-35205.000000,-35504.000000
25%,357.000000,237.000000
50%,488.000000,337.000000
75%,799.000000,574.000000
max,75286.000000,18239.000000


In [22]:
# Look at negative or weird times
df[df["rt_received"] < 0]

,call_number,unit_id,incident_number,call_type,call_date,watch_date,received_dttm,entry_dttm,dispatch_dttm,response_dttm,...,unit_sequence_in_call_dispatch,fire_prevention_district,supervisor_district,neighborhoods_analysis_boundaries,rowid,case_location,data_as_of,data_loaded_at,rt_received,rt_dispatch
42448,230441640,71,23021308,Medical Incident,2023-02-13T00:00:00.000,2023-02-13T00:00:00.000,2023-02-13 13:49:45,2023-02-13T13:51:13.000,2023-02-13 13:54:44,2023-02-13T13:54:47.000,...,1,2.0,5.0,Western Addition,230441640-71,POINT (-122.42383 37.78108),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000,-35205.0,-35504.0
80346,230820452,78,23040233,Medical Incident,2023-03-23T00:00:00.000,2023-03-22T00:00:00.000,2023-03-23 06:26:26,2023-03-23T06:28:33.000,2023-03-23 06:29:28,2023-03-23T06:29:33.000,...,1,1.0,3.0,Chinatown,230820452-78,POINT (-122.41045 37.798374),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000,-1466.0,-1648.0


In [24]:
# Explore geography distribution
df["battalion"].value_counts().head()

battalion
B02    20427
B03    17198
B04    10754
B01     9930
B10     7934
Name: count, dtype: int64

In [25]:
df["zipcode_of_incident"].value_counts().head()

zipcode_of_incident
94103.0    14507
94102.0    12866
94109.0    10416
94110.0     7595
94124.0     4878
Name: count, dtype: int64

In [26]:
# Explore call types
df["call_type_group"].value_counts()

call_type_group
Potentially Life-Threatening    49775
Alarm                           22600
Non Life-threatening            21838
Fire                             4741
Name: count, dtype: int64

In [27]:
# Check missing patterns
df[["on_scene_dttm", "transport_dttm", "hospital_dttm"]].isna().mean()

on_scene_dttm     0.20783
transport_dttm    0.76180
hospital_dttm     0.76410
dtype: float64

In [28]:
# Time patterns (simple version)
df["hour"] = df["received_dttm"].dt.hour

In [29]:
df["hour"].value_counts().sort_index()

hour
0     3087
1     2825
2     2654
3     2041
4     1998
5     1918
6     2229
7     3250
8     4068
9     5017
10    5436
11    5622
12    5677
13    5759
14    5588
15    5605
16    5542
17    5430
18    5306
19    4981
20    4594
21    4206
22    3791
23    3376
Name: count, dtype: int64

In [30]:
# First look at delays (rough idea)
df["rt_received"].quantile([0.5, 0.9, 0.95, 0.99])

0.50     488.00
0.90    1320.00
0.95    1759.00
0.99    3350.84
Name: rt_received, dtype: float64